In [1]:
import os
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!chmod +x datasets
#Update path to include datasets
os.environ['PATH'] += ':/content/datasets:'

!pip install biopython
!datasets --version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18.2M  100 18.2M    0     0  27.9M      0 --:--:-- --:--:-- --:--:-- 27.9M
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 24.3 MB/s eta 0:00:00
datasets version: 18.6.0


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)
%cd drive
%cd MyDrive
%ls
%run GeneClassesCloud.ipynb

!../../datasets --version
#import cmapPy
from time import sleep
import re
import sys
import pandas as pd
import pickle
import os
from collections import defaultdict
from io import BytesIO, StringIO
from pathlib import Path
from zipfile import ZipFile
#from GeneClasses import NaturalGene, ProteinObj, Isoform, IsoformGeneBody
#import cmapPy
#from cmapPy.set_io.grp import read
import numpy as np
from Bio import Entrez
from Bio import SeqIO
from Bio import pairwise2
from Bio.SeqRecord import SeqRecord
import json
import shutil
import dataclasses

Mounted at /content/drive
/content/drive
/content/drive/MyDrive
 alignedHSV1
 AnalysisOfWangDataSet.xlsx
 Autoimmune_diseases_autoantigens_and_incidences.xlsx
 BVBRC_genome.csv
 celltypistmodels/
 CoCoPutsWholeBlood.xlsm
'Colab Notebooks'/
 condacolab_install.log
 datasets
 disease_df.xlsx
 dwnld/
 ERR2882510ReadsPerGene.out.tab
 ERR2882511ReadsPerGene.out.tab
 ERR2882512ReadsPerGene.out.tab
 ERR2882513ReadsPerGene.out.tab
 ERR2882514ReadsPerGene.out.tab
 ERR2882516ReadsPerGene.out.tab
 ERR2882517ReadsPerGene.out.tab
 eTACs.csv
'Executive Summary.gdoc'
 exonTraining/
 firstPassPlusSpliceAI/
 GeneClassesCloud.ipynb
 GeneClasses.py
 GeneLit/
 GeneRegulationNotes.docx
 GeneRider.pptx
 gene_system_pHMMer.ipynb
 gene_table.xlsx
 Genewriter.AI.gdoc
 GreenListPresRefs.docx
 HerpetiQRInvestorPresentationAug2024.pptx
 HerpetiQRInvestorPresentationJul2024.pptx
 HerpetiQRMafft.ipynb
 HerpetiQRmiRNAPrep.ipynb
 HerpetiQRpHMM.ipynb
 HerpetiQR.pptx
 HQ1.pptx.xlsx
 HSV1AllelesHMM/
 HSV1GeneDecisionCha

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


These are the functions we'll use to pull data from NCBI

In [3]:
def check_cds_against_protein(cds, atgs, aa_seq):
    '''Checks that the codons after a given ATG do indeed encode the associated protein. Also returns the codon list'''
    aaseq = [x[0] for x in aa_seq]
    aaCodonVecs = {'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
                   'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
                   'C': ['TGT', 'TGC'],
                   'W': ['TGG'],
                   'E': ['GAA', 'GAG'],
                   'D': ['GAT', 'GAC'],
                   'P': ['CCT', 'CCC', 'CCA', 'CCG'],
                   'V': ['GTT', 'GTC', 'GTA', 'GTG'],
                   'N': ['AAT', 'AAC'],
                   'M': ['ATG'],
                   'K': ['AAA', 'AAG'],
                   'Y': ['TAT', 'TAC'],
                   'I': ['ATT', 'ATC', 'ATA'],
                   'Q': ['CAA', 'CAG'],
                   'F': ['TTT', 'TTC'],
                   'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
                   'T': ['ACT', 'ACC', 'ACA', 'ACG'],
                   '*': ['TAA', 'TAG', 'TGA'],
                   'A': ['GCT', 'GCC', 'GCA', 'GCG'],
                   'G': ['GGT', 'GGC', 'GGA', 'GGG'],
                   'H': ['CAT', 'CAC']}
    for atg in atgs:
      codvec = []
      tf = False
      #Verify that the first codon is indeed ATG
      if cds[atg:atg+3] != 'ATG':
          continue
      #Now loop through the aaseq
      for i in range(len(aa_seq)):
        cod = cds[atg+3*i:atg+3*i+3]
        codvec.append(cod)
        for aa, cods in aaCodonVecs.items():
          if cod in cods:
            aa_from_cds = aa
            break
        aa_from_seq = aaseq[i] #aa_seq[i]
        if aa_from_cds != aa_from_seq:
          continue
        if i == len(aaseq) - 1: #if i == len(aa_seq) - 1:
          tf = True
          break
      if tf:
        return atg, codvec
    return None, None # Return None if no valid ATG is found

def determineProtWeight(aaSeq: str):
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    aaSeq = aaSeq.replace("X", "A")
    prot = ProteinAnalysis(aaSeq)
    return prot.molecular_weight()

def clearDownloadDirectory(dest: str):
  if dest in os.getcwd():
    for f in os.listdir():
      if os.path.isfile(f):
        os.remove(f)
      else:
        shutil.rmtree(f)
      #os.remove(os.path.join(dest, f))
  else:
    print("What are tryna delete?")


def saveNaturalGeneObj(obj: NaturalGene):

    print("CWD: ", str(os.getcwd()))
    while "MyDrive" in os.getcwd():
      %cd ..
    %cd MyDrive
    %cd RefGenes
    %cd NHGeneBodySupp

    name = str(obj.geneID) + r'.json'
    #root = r'RefGenes/NHGenes'
    #fname = os.path.join(root, name)
    fname = name
    #print(fname)
    #print("Beginning json dumps of natural gene")
    #for iso in obj.isoforms:
      #assocProtJSON = dataclasses.asdict(iso.associatedProtein)
      #for exon in iso.geneBody:
      #  exonJSON = dataclasses.asdict(exon)
      #print("Can we do it?")
      #isoAsDict = dataclasses.asdict(iso)
      #print(isoAsDict)
      #isoAsDict['associatedProtein'] = assocProtJSON
      #isoAsDict['geneBody'] = exonJSON


    #raise NotImplementedError

    b = json.dumps(dataclasses.asdict(obj))
    print(b)
    with open(fname, 'wb') as infile:
        infile.write(b.encode('utf-8'))
        #Findme
    infile.close()
    print("# Closed")
    %cd ..
    %cd ..
    %cd WorkingFolders
    %cd NCBIDownload

def getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, rnas, genomes, exonBeginEndOrder):
  print("GetSeqFrom started for ", sequenceName)
  for genome in genomes:
    #print(genome)
    if genomicAccessionVersion == genome.id.split(":")[0]:
      #print("FOUND ONE!")
      range_prelim = genome.id.split(":")[1]
      range_prelim = range_prelim.replace("c", "")
      range_prelim = range_prelim.split("-")
      r1 = int(range_prelim[0])
      r2 = int(range_prelim[1])
      #print(r1)
      #print(r2)
      #print("UUUUUU")

      exonsFull = []
      for ex in exonBeginEndOrder:
        rr1 = int(ex['begin'])
        rr2 = int(ex['end'])
        ord = ex['order']

        #print('rr1: ', rr1)
        #print('rr2: ', rr2)
        #print('--')

        if rr1 > rr2:
          rnaDirection = 'minus'
        else:
          rnaDirection = 'plus'
        #print(rnaDirection)

        #This condition works
        if genomicRange['orientation'] == 'plus' and rnaDirection == 'plus':
          scaledBegin = int(rr1) - r1
          scaledEnd = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd]
          #print(q)
          exonsFull.append(q)

        elif genomicRange['orientation'] == 'plus' and rnaDirection == 'minus':
          print("(Case B:)")
          assert abs(r2 - r1) == abs(rr2 - rr1)
          rnaSnipLen = abs(rr2 - rr1)
          scaledBegin = r1 #- int(rr1)
          scaledEnd = r1 + rnaSnipLen #- int(rr2)
          #scaledBegin = r2 - int(rr1)
          #scaledEnd = int(rr2) - r1
          #scaledEnd = r2 - int(rr2)
          #scaledBegin = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd].reverse_complement()
          #w = genome.seq[scaledBegin:scaledEnd]
          #print(genomicAccessionVersion)
          #print(sequenceName)
          #print(scaledBegin)
          #print(scaledEnd)
          #print(q)
          #print(w)
          raise NotImplementedError
          exonsFull.append(q)

        #this condition works
        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'plus':
          #print("(Case C:)")
          scaledBegin =  r1 - int(rr2)
          scaledEnd = r1 - int(rr1) + 1
          #print("(,", scaledBegin, ", ", scaledEnd,  ")")
          q = genome.seq[scaledBegin:scaledEnd]
          #print(q)
          exonsFull.append(q)

        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'minus':
          print("(Case D:)")
          raise NotImplementedError
          scaledBegin = int(rr1) - r1
          scaledEnd = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd]
          print(q)
          exonsFull.append(q)


  for rna in rnas:
    if genomicAccessionVersion == rna.id:
      print("FOUND ONE! ***")

  return exonsFull

def printNatGene(ng: NaturalGene):
  print(type(ng.geneID))
  print(type(ng.geneName))
  print(type(ng.organism))
  print(type(ng.DNASequence))
  print(type(ng.chromosome))
  print(type(ng.spliceAIDonor))
  print(type(ng.spliceAIReceptor))
  print("%%%")
  for iso in ng.isoforms:
    print(type(iso.isoformNumber))
    print(type(iso.associatedProtein))
    print(type(iso.fullSequence))
    print(type(iso.codingSeq))
    print(type(iso.relativeAbundance))
    print(type(iso.geneBody))
    for ex in iso.geneBody:
      print("LLLLLLLLLLLL")
      print(type(ex['seq']))
      print(type(ex['distFromStart']))
      print(type(ex['distFromEnd']))
    print("$$$")


def exonLocations(geneObj : NaturalGene):
  '''Adds the beginning and end numbers to the NaturalGene's exons in the isoform's gene body'''
  print("Locating exons in DNA")
  for iso in geneObj.isoforms:
    print(iso.isoformNumber)
    for exon in iso.geneBody:
      try:
        #Try to simply find the substring of the exon in the gseq
        exon['start'] = geneObj.DNASequence.index(exon['seq'])
        exon['end'] = exon['start'] + len(exon['seq'])


      except:
        #get left and right values of the exon in the DNASeq by doing an alignment
        print(exon)
        alignments = pairwise2.align.localms(geneObj.DNASequence, exon['seq'], 2, 1, -1, 0)
        #sort alignments by score
        alignments.sort(key=lambda x: x.score)
        alignment = alignments[0]
        print("Len DNA Seq")
        print(len(geneObj.DNASequence))
        print("Start" , alignment[3])
        print("End", alignment[4])
        print("Length of exon: ", len(exon['seq']))
        print("Diff in start and end: ", abs(alignment[3]-alignment[4]) )
        print("&&&&&&")
        #raise NotImplementedError
        exon['start'] = alignment[3]
        exon['end'] = alignment[4]
  return geneObj

def locate_codons(codvec, codingSeq, atg_start, dna_seq, exsupp):
  clocs = []
  eseq = ''
  for e in exsupp:
    s = e['seq']
    eseq += s
  assert eseq == codingSeq

  r = atg_start
  cnum = 0
  carry = ''
  for e in exsupp:
    elen = len(e['seq'])
    if r > len(e['seq']):
      r -= len(e['seq'])
      continue

    ## pick up remainder of the codon from carry
    if carry != '':
      r = 3 - len(carry)
      cod = carry + e['seq'][:r]
      remseq = e['seq'][r:]
      carry = ''
      clocs.append((cod, 'S'))
      if cod == 'TAA' or cod == 'TAG' or cod == 'TGA':
        break
      cnum += 1
      r = 0


    remseq = e['seq'][r:]
    r = 0
    #Add five prime exon end codons if the exon is under 15 longer than remseq
    if elen - len(remseq) < 15:
      while elen - len(remseq) < 15:
        clocs.append((remseq[:3], 'F'))
        remseq = remseq[3:]
        cnum += 1
    #Add internal codons if the remaining seq is over 15 long
    if len(remseq) > 15:
      while len(remseq) > 15:
        clocs.append((remseq[:3], 'I'))
        remseq = remseq[3:]
        cnum += 1
    #Add three prime codons if the remseq is under 15 long
    if len(remseq) <= 15:
      while len(remseq) != 0:
        if len(remseq) < 3:
          carry += remseq
          remseq = ''
          continue
        else:
          clocs.append((remseq[:3], 'T'))
          if len(remseq) == 3 :
            remseq = ''
          else:
            remseq = remseq[3:]
          cnum += 1
  print(clocs)
  return clocs

def downloadGenePackagesAndProcess(gene_ids, data_directory):
    '''Download a gene package and put it into a NaturalGene object. Save to RefGenes/NHGenes'''
    #Uses a method found here: https://sundararamanp.medium.com/a-relatively-faster-approach-for-reading-json-lines-file-into-pandas-dataframe-90b57353fd38
    #print(os.getcwd())

    for gene_id in gene_ids:
      %ls
      #Fix this to suppress errors
      if "NCBIDownload" not in os.getcwd():
        %cd $data_directory
        %cd NCBIDownload
      fnamezip = str(gene_id) + r'.zip'
      fname = str(gene_id)
      #print(fnamezip)
      #print("RRR")
      #print(fname)
      argg = r'gene,rna,protein,cds,3p-utr,5p-utr,product-report'
      #arrg = r'product-report'
      !../../../../datasets download gene gene-id $gene_id --filename $fnamezip --include $argg --no-progressbar
      sleep(5)
      !unzip -o $fnamezip -d $fname
      #cd to the gene data package
      #data_path = os.path.join(data_directory, fname, 'ncbi_dataset', 'data') # Construct the correct path
      # %cd $data_path # No need to change directory, use the full path instead

      # Check if the file exists before attempting to read
      rna_path = os.path.join(fname, 'ncbi_dataset', 'data', 'rna.fna')
      data_path = os.path.join(fname, 'ncbi_dataset', 'data', 'data_report.jsonl')
      pro_path = os.path.join(fname, 'ncbi_dataset', 'data', 'protein.faa')
      futr_path = os.path.join(fname, 'ncbi_dataset', 'data', '5p_utr.fna')
      tutr_path = os.path.join(fname, 'ncbi_dataset', 'data', '3p_utr.fna')
      cds_path = os.path.join(fname, 'ncbi_dataset', 'data', 'cds.fna')
      gene_path = os.path.join(fname, 'ncbi_dataset', 'data', 'gene.fna')
      prodrep_path = os.path.join(fname, 'ncbi_dataset', 'data', 'product_report.jsonl')

      #print("Checking RNA")
      if os.path.exists(rna_path):
          # Use the full path to the RNA file
          rnas = []
          for record in SeqIO.parse(rna_path, 'fasta'):
              #print(record.seq)
              #print(record.id)
              #print(record.description)
              rnas.append(record)
      else:
          print("Error: 'rna.fna' not found in the directory.")

      #print(r"Checking 5'UTR")
      if os.path.exists(futr_path):
          # Use the full path to the RNA file
          futrs = []
          for record in SeqIO.parse(futr_path, 'fasta'):
              #print(record.seq)
              futrs.append(record)
      else:
          print("Error: '5p_utr.fna' not found in the directory.")

      #print(r"Checking cds")
      if os.path.exists(cds_path):
          # Use the full path to the RNA file
          cdss = []
          for record in SeqIO.parse(cds_path, 'fasta'):
              #print(record.seq)
              cdss.append(record)
      else:
          print("Error: 'cds.fna' not found in the directory.")

      #print(r"Checking 3'UTR")
      if os.path.exists(tutr_path):
          # Use the full path to the RNA file
          tutrs = []
          for record in SeqIO.parse(tutr_path, 'fasta'):
              #print(record.seq)
              tutrs.append(record)
      else:
          print("Error: '3p_utr.fna' not found in the directory.")

      #print("Checking proteins")
      if os.path.exists(pro_path):
          # Use the full path to the protein file
          proteins = []
          for record in SeqIO.parse(pro_path, 'fasta'):
              #print(record.seq)
              proteins.append(record)
      else:
          print("Error: 'protein.faa' not found in the directory.")

      #print("Checking genome")
      if os.path.exists(gene_path):
          # Use the full path to the protein file
          genomes = []
          for record in SeqIO.parse(gene_path, 'fasta'):
              #print(record.seq)
              genomes.append(record)
      else:
          print("Error: 'gene.fna' not found in the directory.")

      #print("Checking JSON")
      if os.path.exists(data_path):
          #print("Found JSONL")
          # Use the full path to the JSON file
          with open(data_path) as f:
            lines = f.read().splitlines()
          # Apply json.loads to each line individually
          data = [json.loads(line) for line in lines]
          df_final = pd.json_normalize(data)
          #display(df_final)
      else:
          print("Error: 'data_report.jsonl' not found in the directory.")


      #print("Checking product report")
      if os.path.exists(prodrep_path):
          #print("Found product report")
          with open(prodrep_path) as f:
            lines = f.read().splitlines()
          # Apply json.loads to each line individually
          data1 = [json.loads(line) for line in lines]
          df_final1 = pd.json_normalize(data1)
          #display(df_final1)
      else:
          print("Error: 'product_report.jsonl' not found in the directory.")

      print("Beginning processing for gene " + str(gene_id))

      gSeq = str(genomes[0].seq)
      geneName = df_final['symbol'][0]
      desc = df_final['description'][0]
      organism = df_final['commonName'][0]
      chromosome = df_final['chromosomes'][0]

      if 'synonyms' in df_final:
        synonyms = df_final['synonyms'][0]
      else:
        synonyms = None
      try:
        isoforms = df_final1['transcripts'][0]
      except:
        continue
      try:
        orient = df_final['orientation'][0]
      except:
        display(df_final)
        raise NotImplementedError
      isoSet = []
      exonSets = []
      for iso in isoforms:
        #print("iso")
        #print(iso)


        if 'name' in iso:
          isoformNumber = iso['name'].replace('transcript variant ', '')
        else:
          isoformNumber = -1

        #Transcripts labeled with an X are computationally predicted but not experimentally validated
        try:
          if "X" in isoformNumber:
            print('Skipping ', isoformNumber, " because it isn't exprimentally validated")
            continue
        except TypeError:
            pass

        if len(isoforms) == 1:
          isoformNumber = 1
        #print("ISOFORM: ", isoformNumber)

        if 'protein' in iso:
          tmp = iso['protein']
          protFile = tmp['accessionVersion']

          for p in proteins:
            if protFile in p.id:
              pseq = p.seq
              pweight = determineProtWeight(str(pseq))
              associatedProtein = ProteinObj(str(pseq), pweight, gene_id)
        else:
          associatedProtein = ProteinObj(str(""), 0.0, gene_id)

          #associatedProtein = iso['protein']
        geneAccession = iso['accessionVersion']
        if 'cds' in iso:
          cdsAccession = iso['cds']['accessionVersion']
          cds_temp = iso['cds']
          rng_temp = cds_temp['range']
          rng_temp = rng_temp[0]
          cdsRange = [rng_temp['begin'], rng_temp['end']]
        else:
          print("During json processing, 'CDS' not found. Setting cds range to [-1,-1]")
          cdsRange = [-1,-1]
        if 'ensemblTranscript' in iso:
          ensemblTranscript = iso['ensemblTranscript']
        geneLocs = iso['genomicLocations']

        print("There are ", str(len(geneLocs)), " genomic location sets in ", geneName, " : isoform ", isoformNumber)
        for gLocSet in geneLocs:
          if 'exons' not in gLocSet:
            print("Skipping this one, no exons")
            continue
          #print("LOC!!!")
          #print(gLocSet)
          exonBeginEndOrder = gLocSet['exons']
          genomicAccessionVersion = gLocSet['genomicAccessionVersion']
          genomicRange = gLocSet['genomicRange']
          sequenceName = gLocSet['sequenceName']
          #print("** ** ", genomicAccessionVersion)
          #print("SQ!!!")

          exons = getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, rnas, genomes, exonBeginEndOrder)
          exSupp = []
          cds_seq_from_exons = '' # Initialize cds_seq_from_exons for this exon set
          for ex in exons:
            #print(ex)
            distFromStart = list(range(0, len(ex)))
            exSupp.append({"seq": str(ex), "distFromStart": str(distFromStart)})
            cds_seq_from_exons += str(ex) # Build cds_seq_from_exons
          #exonSets.append(exSupp) # Append exSupp to exonSets if needed later, but not for this fix

          # Now process codingSeq and call locate_codons within this loop
          # This assumes that each gLocSet with exons corresponds to a potential coding sequence
          relativeAbundance = -1
          if associatedProtein.aaSeq != '':
            #Find 'ATG's in codingSeq (use cds_seq_from_exons here)
            atg_indexes = [match.start() for match in re.finditer('ATG', cds_seq_from_exons)]
            atg_start, codvec = check_cds_against_protein(cds_seq_from_exons, atg_indexes, associatedProtein.aaSeq)

            # Check if atg_start or codvec is None before proceeding
            if atg_start is not None and codvec is not None:
                creg = cds_seq_from_exons[atg_start:atg_start+3*len(associatedProtein.aaSeq)]
                print("Locating codons")
                codlocvec = locate_codons(codvec, cds_seq_from_exons, atg_start, gSeq, exSupp) # Pass the correct exSupp and cds_seq_from_exons
                #Write locate_codons!!!

                                                                #Used to be gSeq
                                                                      #
                igb = IsoformGeneBody(isoformNumber, associatedProtein, gSeq, cds_seq_from_exons, codlocvec, relativeAbundance, exSupp) # Pass the correct cds_seq_from_exons and exSupp
                isoSet.append(igb)
            else:
                print(f"Warning: Could not find a valid ATG start codon for gene {gene_id}, isoform {isoformNumber}, coding sequence {cds_seq_from_exons}. Skipping this sequence.")

        # The following loops for refining isoSet should be outside the gLocSet loop
      #print("GGGGGGGGGGG", len(isoSet))
      refinedIsos = []
      for iso in isoSet:
        if iso.geneBody != []:
          #Eliminate non-unique isoforms
          if iso not in refinedIsos:
            refinedIsos.append(iso)
      isoSet = refinedIsos
      #print("UUUUUUUUUU", len(isoSet))
      refinedIsos = []
      protSeqs = []
      for iso in isoSet:
        if iso.geneBody != []:
          #Eliminate non-unique isoforms
          if str(iso.associatedProtein.aaSeq) not in protSeqs:
            protSeqs.append(str(iso.associatedProtein.aaSeq))
            refinedIsos.append(iso)
      isoSet = refinedIsos
      #print("MMMMMMMMMMMMM", len(isoSet))
      refinedIsos = []
      exonSeqs = []
      for iso in isoSet:
        if iso.geneBody != []:
          #Eliminate non-unique isoforms
          linked = ''
          for exon in iso.geneBody:
            linked += exon['seq']
          if linked not in exonSeqs:
            exonSeqs.append(linked)
            refinedIsos.append(iso)
          #isoSet = refinedIsos
      #print("XXXXXXXXXXXXX", len(isoSet))
      #print("XXXXXXXXXXXXX", len(refinedIsos))
      print("!@#$%")
      ng = NaturalGene(isoSet, gene_id, geneName, organism, gSeq, [], [], {}, chromosome)
      print("Exonizing")
      #ng = exonLocations(ng)
      #printNatGene(ng)
      print("Exonized")


      #raise NotImplementedError
      saveNaturalGeneObj(ng)
      #Clear NCBIDownloads
      clearDownloadDirectory(r'NCBIDownload')
      print("Done with gene " + str(gene_id))

Get back to the root directory

In [4]:
#downloadGenePackagesAndProcess([1, 326, 7852, 338, 3586], r'WorkingFolders')
while "RefGenes" in os.getcwd():
  %cd ..
#%cd RefGenes
#%cd NHGeneBodySupp



DATA SOURCING: This is where we'll create our non-housekeeping gene list

List of all human genes:

In [5]:
#https://osf.io/mhda7/
allGenes = []
if os.path.exists('GeneLit/Gene_List.json'):
  with open('GeneLit/Gene_List.json', 'r') as f:
    allGenes = json.load(f)
  print("Gene List loaded            ", "        ----         ", "        ")
else:
  geneTable = pd.read_excel(r'GeneLit/Gene_Table.xlsx', usecols = [0])
  for r in geneTable.index:
    a = str(geneTable['Gene_ID'][r])
    if a not in allGenes:
      allGenes.append(a)
  geneTable = None
  del geneTable
  print("Gene List loaded            ", "        ----         ", "        ")
  #Save allGenes list
  with open('GeneLit/Gene_List.json', 'w') as f:
    json.dump(allGenes, f)
print("Length of AllGenes: ", len(allGenes))

Gene List loaded                     ----                  
Length of AllGenes:  19106


List of human housekeeping genes

In [6]:
#Define a list of housekeeping genes
#https://www.gsea-msigdb.org/gsea/msigdb/cards/HSIAO_HOUSEKEEPING_GENES
HCGenes = []
if os.path.exists('GeneLit/Housekeeping_Genes.json'):
  with open('GeneLit/Housekeeping_Genes.json', 'r') as f:
    HCGenes = json.load(f)
  print("Housekeeping Genes loaded            ", "        ----         ", "        ")
else:
  !pip install cmapPy
  from cmapPy.set_io.grp import read
  houseGenes = read('GeneLit/HSIAO_HOUSEKEEPING_GENES.v2023.2.Hs.grp')
  t = 0
  HCGenes = houseGenes[1:]
  #Save HCGenes list
  with open('GeneLit/Housekeeping_Genes.json', 'w') as f:
    json.dump(HCGenes, f)
  del houseGenes
  print("Housekeeping Genes loaded            ", "        ----         ", "        ")

print("Length of HCGenes: ", len(HCGenes))
print(HCGenes[:10])

Housekeeping Genes loaded                     ----                  
Length of HCGenes:  397
['AAMP', 'AARS1', 'ABLIM1', 'ACKR1', 'ACTB', 'ACTG1', 'AGPAT1', 'ALDOA', 'ANP32B', 'ANXA11']


Subtract out the housekeeping genes

In [7]:
#Create a list of non-housekeeping genes
NHGenes = []
for ag in allGenes:
  if ag not in HCGenes:
    NHGenes.append(int(ag))

del allGenes
del HCGenes
print("Length of NHGenes: ", len(NHGenes))
print(NHGenes[:10])

Length of NHGenes:  19106
[5747, 7157, 7422, 1956, 627, 6935, 23353, 2908, 348, 4292]


Subtract out genes we've already processed

In [8]:
#Prune list if we already have the files
dir = os.getcwd()
print(dir)
dir2 = os.path.join(dir, r'RefGenes', r'NHGeneBodySupp')
print(dir2)
files = os.listdir(dir2)
files = [f for f in files if os.path.isfile(os.path.join(dir2, f))]
files = [f for f in files if '.json' in f]
files.sort()
print(len(files))
i = 0
while i < len(files):
  files[i] = int(files[i].replace('.json', ''))
  i += 1
NHGenes= [x for x in NHGenes if x not in files]
print(len(NHGenes))

#Increment this to skip genes which require too much RAM to seperate exons from
y = 242
print(y)
NHGenes = NHGenes[y:]

/content/drive/MyDrive
/content/drive/MyDrive/RefGenes/NHGeneBodySupp
18864
242
241


Now, run the downloadGenePackagesAndProcess procedure.

In [9]:
#Pull all non-housekeeping genes from NCBI
#downloadGenePackages(NHGenes, 'WorkingFolders')


downloadGenePackagesAndProcess(NHGenes, r'WorkingFolders')

 alignedHSV1
 AnalysisOfWangDataSet.xlsx
 Autoimmune_diseases_autoantigens_and_incidences.xlsx
 BVBRC_genome.csv
 celltypistmodels/
 CoCoPutsWholeBlood.xlsm
'Colab Notebooks'/
 condacolab_install.log
 datasets
 disease_df.xlsx
 dwnld/
 ERR2882510ReadsPerGene.out.tab
 ERR2882511ReadsPerGene.out.tab
 ERR2882512ReadsPerGene.out.tab
 ERR2882513ReadsPerGene.out.tab
 ERR2882514ReadsPerGene.out.tab
 ERR2882516ReadsPerGene.out.tab
 ERR2882517ReadsPerGene.out.tab
 eTACs.csv
'Executive Summary.gdoc'
 exonTraining/
 firstPassPlusSpliceAI/
 GeneClassesCloud.ipynb
 GeneClasses.py
 GeneLit/
 GeneRegulationNotes.docx
 GeneRider.pptx
 gene_system_pHMMer.ipynb
 gene_table.xlsx
 Genewriter.AI.gdoc
 GreenListPresRefs.docx
 HerpetiQRInvestorPresentationAug2024.pptx
 HerpetiQRInvestorPresentationJul2024.pptx
 HerpetiQRMafft.ipynb
 HerpetiQRmiRNAPrep.ipynb
 HerpetiQRpHMM.ipynb
 HerpetiQR.pptx
 HQ1.pptx.xlsx
 HSV1AllelesHMM/
 HSV1GeneDecisionChart.docx
 HSV1_Genomes/
 HSV1_Genomes_Aligned/
 HSV1_Genomes_Stra